# Day 29: Autoregressive Decoder Loop

**Layer:** Implementation | **Prerequisite:** Day 01 (Decoding), Day 28 (Tokenizer)


## Concept Overview

The autoregressive decoder loop is the core of LLM inference: repeatedly sample one token from the model's output distribution, append it to the input, and repeat until the stop condition. The KV cache avoids recomputing attention for previous tokens.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Minimal Transformer for Decoding


In [ ]:
class MinimalAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, kv_cache=None):
        B, T, D = x.shape
        qkv = self.qkv(x).chunk(3, dim=-1)
        q, k, v = [t.view(B, T, self.n_heads, self.d_head).transpose(1,2) for t in qkv]
        if kv_cache is not None:
            k = torch.cat([kv_cache['k'], k], dim=2)
            v = torch.cat([kv_cache['v'], v], dim=2)
        new_cache = {'k': k, 'v': v}
        scale = self.d_head ** -0.5
        attn = F.softmax(q @ k.transpose(-2,-1) * scale, dim=-1)
        out = (attn @ v).transpose(1,2).reshape(B, T, D)
        return self.out(out), new_cache

class MinimalDecoder(nn.Module):
    def __init__(self, vocab_size=1000, d_model=128, n_heads=4, n_layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([MinimalAttention(d_model, n_heads) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.vocab_size = vocab_size

    def forward(self, tokens, kv_caches=None):
        x = self.embed(tokens)
        new_caches = []
        for i, layer in enumerate(self.layers):
            cache = kv_caches[i] if kv_caches else None
            x, new_cache = layer(x, cache)
            new_caches.append(new_cache)
        return self.lm_head(self.norm(x)), new_caches

model = MinimalDecoder()
print(f'Model params: {sum(p.numel() for p in model.parameters()):,}')


## 2. Autoregressive Generation Loop


In [ ]:
def generate(model, prompt_tokens, max_new_tokens=20, temperature=1.0,
             top_k=50, device='cpu'):
    model.eval()
    tokens = torch.tensor([prompt_tokens], device=device)
    generated = []
    kv_caches = None
    ttft_measured = False
    times = []
    t_start = time.perf_counter()

    with torch.no_grad():
        # Prefill: process all prompt tokens at once
        logits, kv_caches = model(tokens)
        if not ttft_measured:
            ttft = (time.perf_counter() - t_start) * 1000
            ttft_measured = True
        next_token = sample_token(logits[:, -1, :], temperature, top_k)
        generated.append(next_token)

        # Decode: one token at a time, using KV cache
        for _ in range(max_new_tokens - 1):
            t0 = time.perf_counter()
            inp = torch.tensor([[next_token]], device=device)
            logits, kv_caches = model(inp, kv_caches)
            next_token = sample_token(logits[:, -1, :], temperature, top_k)
            times.append((time.perf_counter() - t0) * 1000)
            generated.append(next_token)
            if next_token == 2:  # <eos> = 2
                break

    return generated, ttft, np.mean(times) if times else 0

def sample_token(logits, temperature, top_k):
    logits = logits / temperature
    top_logits, top_indices = torch.topk(logits, top_k)
    probs = F.softmax(top_logits, dim=-1)
    idx = torch.multinomial(probs, 1)
    return top_indices[0, idx.item()].item()

torch.manual_seed(42)
prompt = [10, 20, 30, 40, 50]
generated, ttft, tpot = generate(model, prompt, max_new_tokens=20)
print(f'Prompt: {prompt}')
print(f'Generated: {generated}')
print(f'TTFT: {ttft:.2f} ms (prefill {len(prompt)} tokens)')
print(f'TPOT: {tpot:.2f} ms (mean decode step)')


## 3. KV Cache Memory Growth


In [ ]:
# Measure KV cache memory vs sequence length
def measure_kv_growth(model, max_len=100):
    tokens = [1] * 10  # start with 10 tokens
    _, kv = model(torch.tensor([tokens]))
    sizes = []
    for i in range(max_len):
        total_bytes = sum(
            kv[l]['k'].nelement() * kv[l]['k'].element_size() +
            kv[l]['v'].nelement() * kv[l]['v'].element_size()
            for l in range(len(kv))
        )
        sizes.append(total_bytes / 1024)  # KB
        next_tok = torch.tensor([[42]])
        _, kv = model(next_tok, kv)
    return sizes

sizes = measure_kv_growth(model)
plt.plot(range(10, 10+len(sizes)), sizes)
plt.xlabel('Sequence length'); plt.ylabel('KV cache size (KB)')
plt.title('KV Cache Memory Growth (Linear in Sequence Length)')
plt.grid(True)
plt.savefig('kv_growth.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'KV cache at seq=10: {sizes[0]:.2f} KB')
print(f'KV cache at seq=110: {sizes[-1]:.2f} KB')
print(f'Linear growth ratio: {sizes[-1]/sizes[0]:.1f}x (10x length = {sizes[-1]/sizes[0]:.1f}x memory)')


## Experiments: Try These

1. Add streaming output: yield tokens as they're generated and measure user-perceived time-to-first-token.
2. Implement beam search on top of this loop. How does it change latency vs greedy?
3. Remove the KV cache and measure the slowdown per token as sequence length grows.


## Key Takeaways

- The autoregressive loop is: prefill once (full prompt in parallel) → decode one token at a time, appending to context.
- KV cache grows linearly with sequence length — this is why context length × batch size × model size determines GPU memory budget.
- Sampling temperature controls output diversity; top-k/top-p pruning prevents low-probability nonsense tokens.
- TTFT is dominated by prefill (quadratic in prompt length); TPOT is dominated by weight loading (linear, memory-bound).

## References
- Brown et al. (2020), "Language Models are Few-Shot Learners"
- HuggingFace generate() documentation
